In [95]:
import pandas as pd

In [96]:
prior_df = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/order_product_data_prior.csv")

In [97]:
print(prior_df.eval_set.value_counts())

eval_set
prior    32434489
Name: count, dtype: int64


In [98]:
print(prior_df.columns)

Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order', 'product_id',
       'add_to_cart_order', 'reordered', 'product_name', 'aisle_id',
       'department_id', 'department'],
      dtype='str')


In [99]:
prior_df.head(10)

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,department
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,beverages
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,dairy eggs
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,snacks
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,snacks
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,household
5,2398795,1,prior,2,3,7,15.0,196,1,1,Soda,77,7,beverages
6,2398795,1,prior,2,3,7,15.0,10258,2,0,Pistachios,117,19,snacks
7,2398795,1,prior,2,3,7,15.0,12427,3,1,Original Beef Jerky,23,19,snacks
8,2398795,1,prior,2,3,7,15.0,13176,4,0,Bag of Organic Bananas,24,4,produce
9,2398795,1,prior,2,3,7,15.0,26088,5,1,Aged White Cheddar Popcorn,23,19,snacks


In [100]:
print(prior_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 14 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                str    
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
 7   product_id              int64  
 8   add_to_cart_order       int64  
 9   reordered               int64  
 10  product_name            str    
 11  aisle_id                int64  
 12  department_id           int64  
 13  department              str    
dtypes: float64(1), int64(10), str(3)
memory usage: 4.5 GB
None


### prior_df data will be used to create user history

In [101]:
print("Unique users:", prior_df["user_id"].nunique())
print("Unique orders:", prior_df["order_id"].nunique())
print("Unique products:", prior_df["product_id"].nunique())

Unique users: 206209
Unique orders: 3214874
Unique products: 49677


In [102]:
df_2 = prior_df.groupby(["order_id", "product_id"]).size()
df_2.head()

order_id  product_id
2         1819          1
          9327          1
          17794         1
          28985         1
          30035         1
dtype: int64

In [103]:
# Whether each row represents a single product for each order
print(prior_df.groupby(["order_id", "product_id"]).size().value_counts())

# whether an order has duplicate proudct_id or not

duplicates = prior_df.groupby(["order_id", "product_id"]).size().reset_index(name='count')
duplicates[duplicates["count"] > 1]

1    32434489
Name: count, dtype: int64


,order_id,product_id,count


In [104]:
prior_df = prior_df.sort_values(["user_id", "order_number", "order_id", "add_to_cart_order"]).reset_index(drop=True)

In [105]:
print(f"The total number of order by user_id 1 is {max(prior_df[prior_df.user_id == 1]['order_number'])}")

The total number of order by user_id 1 is 10


In [106]:
order_context = (
    prior_df.groupby("order_id")
      .agg(
          user_id=("user_id", "first"),
          order_number=("order_number", "first"),
          order_dow=("order_dow", "first"),
          order_hour_of_day=("order_hour_of_day", "first"),
          days_since_prior_order=("days_since_prior_order", "first"),
          eval_set=("eval_set", "first"),
          basket_size=("product_id", "nunique")
      )
      .reset_index()
)

order_context = order_context.sort_values(["user_id", "order_number", "order_id"]).reset_index(drop=True)

In [107]:
order_context.head(12)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,eval_set,basket_size
0,2539329,1,1,2,8,NaN,prior,5
1,2398795,1,2,3,7,15.0,prior,6
2,473747,1,3,3,12,21.0,prior,5
3,2254736,1,4,4,7,29.0,prior,5
4,431534,1,5,4,15,28.0,prior,8
5,3367565,1,6,2,7,19.0,prior,4
6,550135,1,7,1,9,20.0,prior,5
7,3108588,1,8,1,14,14.0,prior,6
8,2295261,1,9,1,16,0.0,prior,6
9,2550362,1,10,4,8,30.0,prior,9


In [108]:
order_baskets = (
    prior_df.sort_values(["order_id", "add_to_cart_order"]).groupby("order_id")
            .agg(
            product_ids=("product_id", list),
            product_names=("product_name", list),
            departments=("department", list),
            cart_sequence=("add_to_cart_order", list)
        ).reset_index()
)

In [109]:
order_baskets.head()

,order_id,product_ids,product_names,departments,cart_sequence
0,2,"[33120, 28985, 9327, 45918, 30035, 17794, 4014...","[Organic Egg Whites, Michigan Organic Kale, Ga...","[dairy eggs, produce, pantry, pantry, pantry, ...","[1, 2, 3, 4, 5, 6, 7, 8, 9]"
1,3,"[33754, 24838, 17704, 21903, 17668, 46667, 174...",[Total 2% with Strawberry Lowfat Greek Straine...,"[dairy eggs, dairy eggs, produce, produce, dai...","[1, 2, 3, 4, 5, 6, 7, 8]"
2,4,"[46842, 26434, 39758, 27761, 10054, 21351, 225...","[Plain Pre-Sliced Bagels, Honey/Lemon Cough Dr...","[bakery, personal care, snacks, breakfast, bre...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]"
3,5,"[13176, 15005, 47329, 27966, 23909, 48370, 132...","[Bag of Organic Bananas, Just Crisp, Parmesan,...","[produce, pantry, deli, produce, dairy eggs, h...","[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
4,6,"[40462, 15873, 41897]","[Cleanse, Dryer Sheets Geranium Scent, Clean D...","[beverages, household, household]","[1, 2, 3]"


In [110]:
user_orders = order_context

In [111]:
user_orders.head(12)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,eval_set,basket_size
0,2539329,1,1,2,8,NaN,prior,5
1,2398795,1,2,3,7,15.0,prior,6
2,473747,1,3,3,12,21.0,prior,5
3,2254736,1,4,4,7,29.0,prior,5
4,431534,1,5,4,15,28.0,prior,8
5,3367565,1,6,2,7,19.0,prior,4
6,550135,1,7,1,9,20.0,prior,5
7,3108588,1,8,1,14,14.0,prior,6
8,2295261,1,9,1,16,0.0,prior,6
9,2550362,1,10,4,8,30.0,prior,9


In [112]:
user_features = (
    user_orders
    .groupby("user_id")
    .agg(
        user_total_orders=("order_id", "nunique"),
        user_avg_basket_size=("basket_size", "mean"),
        user_avg_days_between_orders=("days_since_prior_order", "mean"),
        user_avg_order_hour=("order_hour_of_day", "mean")
    )
    .reset_index()
)

In [113]:
user_features.head(12)

,user_id,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_avg_order_hour
0,1,10,5.900000,19.555556,10.300000
1,2,14,13.928571,15.230769,10.571429
2,3,12,7.333333,12.090909,16.416667
3,4,5,3.600000,13.750000,12.600000
4,5,4,9.250000,13.333333,16.000000
5,6,3,4.666667,9.000000,17.333333
6,7,20,10.300000,10.684211,13.600000
7,8,3,16.333333,30.000000,2.666667
8,9,3,25.333333,18.000000,14.333333
9,10,5,28.600000,19.750000,16.600000


In [114]:
user_product_history = (
    prior_df.groupby(["user_id", "product_id"])
      .agg(
          user_product_orders=("order_id", "nunique"),
          user_product_reorders=("reordered", "sum"),
          user_product_first_order=("order_number", "min"),
          user_product_last_order=("order_number", "max")
      )
      .reset_index()
)

In [115]:
user_product_history.head()

,user_id,product_id,user_product_orders,user_product_reorders,user_product_first_order,user_product_last_order
0,1,196,10,9,1,10
1,1,10258,9,8,2,10
2,1,10326,1,0,5,5
3,1,12427,10,9,1,10
4,1,13032,3,2,2,10


In [116]:
user_product_history["user_product_reorder_rate"] = (user_product_history["user_product_reorders"]
                                                     / 
                                                     user_product_history["user_product_orders"]
)

In [117]:
prior_df["is_weekend"] = prior_df["order_dow"].isin([0, 6]).astype("int8")

In [118]:
def get_time_period(hour):
    if 5 <= hour < 12:
        return "morning"
    elif 12 <= hour < 17:
        return "afternoon"
    elif 17 <= hour < 21:
        return "evening"
    else:
        return "night"

prior_df["time_period"] = prior_df["order_hour_of_day"].apply(get_time_period)

In [119]:
def create_cart_states(order_df):
    order_df = order_df.sort_values("add_to_cart_order")
    products = order_df["product_id"].tolist()
    states = []

    for i in range(1, len(products)):
        current_cart = products[:i]
        target_products = products[i:]

        states.append({
            "order_id": order_df["order_id"].iloc[0],
            "user_id": order_df["user_id"].iloc[0],
            "cart_size": len(current_cart),
            "current_cart": current_cart,
            "target_products": target_products
        })

    return states

In [120]:
example_order = prior_df[prior_df["order_id"] == prior_df["order_id"].iloc[0]]
create_cart_states(example_order)

# how the recommendation should work

[{'order_id': np.int64(2539329),
  'user_id': np.int64(1),
  'cart_size': 1,
  'current_cart': [196],
  'target_products': [14084, 12427, 26088, 26405]},
 {'order_id': np.int64(2539329),
  'user_id': np.int64(1),
  'cart_size': 2,
  'current_cart': [196, 14084],
  'target_products': [12427, 26088, 26405]},
 {'order_id': np.int64(2539329),
  'user_id': np.int64(1),
  'cart_size': 3,
  'current_cart': [196, 14084, 12427],
  'target_products': [26088, 26405]},
 {'order_id': np.int64(2539329),
  'user_id': np.int64(1),
  'cart_size': 4,
  'current_cart': [196, 14084, 12427, 26088],
  'target_products': [26405]}]

In [121]:
# splitting the current cart state into half for training and testing 
def create_training_state(order_df):
    order_df = order_df.sort_values("add_to_cart_order")
    products = order_df["product_id"].tolist()
    
    if len(products) < 2:
        return None

    split_point = len(products) // 2

    current_cart = products[:split_point]
    target_products = products[split_point:]

    return {
        "order_id": order_df["order_id"].iloc[0],
        "user_id": order_df["user_id"].iloc[0],
        "order_number": order_df["order_number"].iloc[0],
        "order_dow": order_df["order_dow"].iloc[0],
        "order_hour_of_day": order_df["order_hour_of_day"].iloc[0],
        "days_since_prior_order": order_df["days_since_prior_order"].iloc[0],
        "cart_size": len(current_cart),
        "current_cart": current_cart,
        "target_products": target_products
    }

In [122]:
example_order = prior_df[prior_df["order_id"] == prior_df["order_id"].iloc[0]]
create_training_state(example_order)

{'order_id': np.int64(2539329),
 'user_id': np.int64(1),
 'order_number': np.int64(1),
 'order_dow': np.int64(2),
 'order_hour_of_day': np.int64(8),
 'days_since_prior_order': np.float64(nan),
 'cart_size': 2,
 'current_cart': [196, 14084],
 'target_products': [12427, 26088, 26405]}

In [123]:
user_orders.to_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/user_orders.csv",index=False)
user_features.to_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/user_features.csv",index=False)

In [124]:
user_product_history.to_pickle("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/user_product_history.pkl")

### train_df will be used to train the model for product recommendation

In [125]:
train_df = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/order_product_data_train.csv")


In [126]:
train_df.head()

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,department
0,1187899,1,train,11,4,8,14.0,196,1,1,Soda,77,7,beverages
1,1187899,1,train,11,4,8,14.0,25133,2,1,Organic String Cheese,21,16,dairy eggs
2,1187899,1,train,11,4,8,14.0,38928,3,1,0% Greek Strained Yogurt,120,16,dairy eggs
3,1187899,1,train,11,4,8,14.0,26405,4,1,XL Pick-A-Size Paper Towel Rolls,54,17,household
4,1187899,1,train,11,4,8,14.0,39657,5,1,Milk Chocolate Almonds,45,19,snacks


In [127]:
print("Prior orders:", prior_df["order_id"].nunique())
print("Train orders:", train_df["order_id"].nunique())

Prior orders: 3214874
Train orders: 131209


In [128]:
train_order_context = (
    train_df
    .groupby("order_id")
    .agg(
        user_id=("user_id", "first"),
        order_number=("order_number", "first"),
        order_dow=("order_dow", "first"),
        order_hour_of_day=("order_hour_of_day", "first"),
        days_since_prior_order=("days_since_prior_order", "first")
    )
    .reset_index()
)

train_order_context = (
    train_order_context
    .sort_values(["user_id", "order_number", "order_id"])
    .reset_index(drop=True)
)

In [129]:
print("TRAIN orders:", train_df["order_id"].nunique())
print("TRAIN product rows:", train_df["product_id"].notna().sum())
print("TRAIN product IDs:", train_df["product_id"].nunique())

TRAIN orders: 131209
TRAIN product rows: 1384617
TRAIN product IDs: 39123


In [130]:
sample_order = (
    train_df[train_df["order_id"] == train_df["order_id"].iloc[0]]
    .sort_values("add_to_cart_order")
)

sample_order[
    [
        "order_id",
        "user_id",
        "order_number",
        "order_dow",
        "order_hour_of_day",
        "product_id",
        "product_name",
        "add_to_cart_order"
    ]
]

,order_id,user_id,order_number,order_dow,order_hour_of_day,product_id,product_name,add_to_cart_order
0,1187899,1,11,4,8,196,Soda,1
1,1187899,1,11,4,8,25133,Organic String Cheese,2
2,1187899,1,11,4,8,38928,0% Greek Strained Yogurt,3
3,1187899,1,11,4,8,26405,XL Pick-A-Size Paper Towel Rolls,4
4,1187899,1,11,4,8,39657,Milk Chocolate Almonds,5
5,1187899,1,11,4,8,10258,Pistachios,6
6,1187899,1,11,4,8,13032,Cinnamon Toast Crunch,7
7,1187899,1,11,4,8,26088,Aged White Cheddar Popcorn,8
8,1187899,1,11,4,8,27845,Organic Whole Milk,9
9,1187899,1,11,4,8,49235,Organic Half & Half,10


### Making the training dataset 

In [131]:
train_order_products = (
    train_df[
        [
            "order_id",
            "user_id",
            "order_number",
            "order_dow",
            "order_hour_of_day",
            "days_since_prior_order",
            "product_id",
            "product_name",
            "add_to_cart_order"
        ]
    ]
    .sort_values(["order_id", "add_to_cart_order"])
    .copy()
)

In [132]:
# Current cart creation 

def create_cart(row, order_group):
    target_position = row["add_to_cart_order"]

    current_cart = order_group[
        order_group["add_to_cart_order"] < target_position
    ]["product_id"].tolist()

    return current_cart

In [133]:
training_states = []

for order_id, order_group in train_order_products.groupby("order_id"):

    order_group = order_group.sort_values("add_to_cart_order")
    for _, target_row in order_group.iterrows():

        current_cart = order_group[
            order_group["add_to_cart_order"] < target_row["add_to_cart_order"]
        ]["product_id"].tolist()

        if len(current_cart) == 0:
            continue

        training_states.append({
            "order_id": order_id,
            "user_id": target_row["user_id"],
            "order_number": target_row["order_number"],
            "order_dow": target_row["order_dow"],
            "order_hour_of_day": target_row["order_hour_of_day"],
            "days_since_prior_order": target_row["days_since_prior_order"],
            "current_cart": current_cart,
            "cart_size": len(current_cart),
            "target_product_id": target_row["product_id"]
        })

training_states = pd.DataFrame(training_states)

In [134]:
example_order = train_df[train_df["order_id"] == 1]
create_training_state(example_order)

{'order_id': np.int64(1),
 'user_id': np.int64(112108),
 'order_number': np.int64(4),
 'order_dow': np.int64(4),
 'order_hour_of_day': np.int64(10),
 'days_since_prior_order': np.float64(9.0),
 'cart_size': 4,
 'current_cart': [49302, 11109, 10246, 49683],
 'target_products': [43633, 13176, 47209, 22035]}

In [135]:
training_states.head(10)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,current_cart,cart_size,target_product_id
0,1,112108,4,4,10,9.0,[49302],1,11109
1,1,112108,4,4,10,9.0,"[49302, 11109]",2,10246
2,1,112108,4,4,10,9.0,"[49302, 11109, 10246]",3,49683
3,1,112108,4,4,10,9.0,"[49302, 11109, 10246, 49683]",4,43633
4,1,112108,4,4,10,9.0,"[49302, 11109, 10246, 49683, 43633]",5,13176
5,1,112108,4,4,10,9.0,"[49302, 11109, 10246, 49683, 43633, 13176]",6,47209
6,1,112108,4,4,10,9.0,"[49302, 11109, 10246, 49683, 43633, 13176, 47209]",7,22035
7,36,79431,23,6,18,30.0,[39612],1,19660
8,36,79431,23,6,18,30.0,"[39612, 19660]",2,49235
9,36,79431,23,6,18,30.0,"[39612, 19660, 49235]",3,43086


In [136]:
training_states.to_pickle("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/training_states.pkl")

In [137]:
print(user_orders.columns)
print(user_features.columns)
print(user_product_history.columns)

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'eval_set', 'basket_size'],
      dtype='str')
Index(['user_id', 'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour'],
      dtype='str')
Index(['user_id', 'product_id', 'user_product_orders', 'user_product_reorders',
       'user_product_first_order', 'user_product_last_order',
       'user_product_reorder_rate'],
      dtype='str')


In [138]:
def create_order_training_state(order_group):

    order_group = order_group.sort_values("add_to_cart_order")
    products = order_group["product_id"].tolist()
    split_point = max(1, len(products) // 2)

    return {
        "order_id": order_group["order_id"].iloc[0],
        "user_id": order_group["user_id"].iloc[0],
        "order_number": order_group["order_number"].iloc[0],
        "order_dow": order_group["order_dow"].iloc[0],
        "order_hour_of_day": order_group["order_hour_of_day"].iloc[0],
        "days_since_prior_order": order_group["days_since_prior_order"].iloc[0],
        "current_cart": products[:split_point],
        "target_products": products[split_point:],
        "cart_size": len(products[:split_point])
    }

In [139]:
order_states = []

for order_id, group in train_df.groupby("order_id"):
    
    state = create_order_training_state(group)
    if len(state["target_products"]) > 0:
        order_states.append(state)

order_states = pd.DataFrame(order_states)

In [140]:
order_states.head(10)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,current_cart,target_products,cart_size
0,1,112108,4,4,10,9.0,"[49302, 11109, 10246, 49683]","[43633, 13176, 47209, 22035]",4
1,36,79431,23,6,18,30.0,"[39612, 19660, 49235, 43086]","[46620, 34497, 48679, 46979]",4
2,38,42756,6,6,16,24.0,"[11913, 18159, 4461, 21616]","[23622, 32433, 28842, 42625, 39693]",4
3,96,17227,7,6,20,30.0,"[20574, 30391, 40706]","[25610, 27966, 24489, 39275]",3
4,98,56463,41,3,8,14.0,"[8859, 19731, 43654, 13176, 4357, 37664, 34065...","[27344, 47333, 48287, 45204, 24964, 18117, 464...",24
5,112,125030,5,5,14,26.0,"[27104, 21174, 41860, 38273, 47209]","[5876, 29217, 9047, 4549, 22425, 11776]",5
6,170,182389,7,0,13,14.0,"[18394, 37766, 13176, 6236, 5077, 8153, 43772,...","[34582, 49593, 15093, 43841, 21137, 40354, 177...",8
7,218,98711,12,0,21,17.0,"[1194, 5578]","[38159, 10305, 38557]",2
8,226,51011,4,0,12,30.0,"[28199, 24852, 29883, 28427, 7754, 39947]","[47307, 36291, 39275, 1940, 2040, 20711, 47501]",6
9,349,156353,9,3,16,30.0,"[33000, 11361, 27695, 47672, 45633]","[38015, 36968, 30830, 5115, 11520, 25715]",5


In [141]:
order_states.to_pickle("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/order_training_states.pkl")

In [142]:
# global popular product 
popular_products = (prior_df.groupby("product_id").size().sort_values(ascending=False).head(100).index.tolist())
print(len(popular_products))

100


In [143]:
# product history candidate for each user 

user_history_candidates = (
    user_product_history
    .sort_values(["user_id", "user_product_orders"], ascending=[True, False])
    .groupby("user_id")
    .head(30)
)

user_history_candidates.head(10)

,user_id,product_id,user_product_orders,user_product_reorders,user_product_first_order,user_product_last_order,user_product_reorder_rate
0,1,196,10,9,1,10,0.900000
3,1,12427,10,9,1,10,0.900000
1,1,10258,9,8,2,10,0.888889
8,1,25133,8,7,3,10,0.875000
4,1,13032,3,2,2,10,0.666667
16,1,46149,3,2,8,10,0.666667
5,1,13176,2,1,2,5,0.500000
9,1,26088,2,1,1,2,0.500000
10,1,26405,2,1,1,4,0.500000
17,1,49235,2,1,8,9,0.500000


### Generate training candidate dataset

In [144]:
history_lookup = (user_product_history.groupby("user_id")["product_id"].apply(list).to_dict())

candidate_rows = []

for _, row in order_states.iterrows():

    user_id = row["user_id"]

    current_cart = set(row["current_cart"])
    target_products = set(row["target_products"])

    history_candidates = history_lookup.get(user_id, [])
    candidates = set(history_candidates[:30])
    candidates.update(popular_products[:20])

    # Don't recommend something already in cart
    candidates = candidates - current_cart

    for product_id in candidates:

        candidate_rows.append({
            "order_id": row["order_id"],
            "user_id": user_id,
            "order_number": row["order_number"],
            "order_dow": row["order_dow"],
            "order_hour_of_day": row["order_hour_of_day"],
            "days_since_prior_order": row["days_since_prior_order"],
            "cart_size": row["cart_size"],
            "product_id": product_id,
            "target": int(product_id in target_products)
        })

candidate_df = pd.DataFrame(candidate_rows)

In [145]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target'],
      dtype='str')

In [146]:
print(candidate_df.shape)
print(candidate_df["target"].value_counts())
candidate_df.head()

(5334991, 9)
target
0    5124089
1     210902
Name: count, dtype: int64


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,cart_size,product_id,target
0,1,112108,4,4,10,9.0,4,40706,0
1,1,112108,4,4,10,9.0,4,24964,0
2,1,112108,4,4,10,9.0,4,27845,0
3,1,112108,4,4,10,9.0,4,44359,0
4,1,112108,4,4,10,9.0,4,47626,0


In [147]:
# adding candidate level feature

candidate_df = candidate_df.merge(user_features, on="user_id", how="left")
candidate_df.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,cart_size,product_id,target,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_avg_order_hour
0,1,112108,4,4,10,9.0,4,40706,0,3,7.0,11.0,15.0
1,1,112108,4,4,10,9.0,4,24964,0,3,7.0,11.0,15.0
2,1,112108,4,4,10,9.0,4,27845,0,3,7.0,11.0,15.0
3,1,112108,4,4,10,9.0,4,44359,0,3,7.0,11.0,15.0
4,1,112108,4,4,10,9.0,4,47626,0,3,7.0,11.0,15.0


In [148]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target',
       'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour'],
      dtype='str')

In [149]:
# add user product history

candidate_df = candidate_df.merge(user_product_history, on=["user_id", "product_id"], how="left" )
candidate_df.head()

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,cart_size,product_id,target,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_avg_order_hour,user_product_orders,user_product_reorders,user_product_first_order,user_product_last_order,user_product_reorder_rate
0,1,112108,4,4,10,9.0,4,40706,0,3,7.0,11.0,15.0,NaN,NaN,NaN,NaN,NaN
1,1,112108,4,4,10,9.0,4,24964,0,3,7.0,11.0,15.0,NaN,NaN,NaN,NaN,NaN
2,1,112108,4,4,10,9.0,4,27845,0,3,7.0,11.0,15.0,NaN,NaN,NaN,NaN,NaN
3,1,112108,4,4,10,9.0,4,44359,0,3,7.0,11.0,15.0,2.0,1.0,1.0,2.0,0.5
4,1,112108,4,4,10,9.0,4,47626,0,3,7.0,11.0,15.0,NaN,NaN,NaN,NaN,NaN


In [150]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target',
       'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour',
       'user_product_orders', 'user_product_reorders',
       'user_product_first_order', 'user_product_last_order',
       'user_product_reorder_rate'],
      dtype='str')

In [151]:
history_cols = ["user_product_orders", "user_product_reorders", 
                "user_product_first_order", "user_product_last_order", 
                "user_product_reorder_rate"]

candidate_df[history_cols] = (candidate_df[history_cols].fillna(0))

In [152]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target',
       'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour',
       'user_product_orders', 'user_product_reorders',
       'user_product_first_order', 'user_product_last_order',
       'user_product_reorder_rate'],
      dtype='str')

In [153]:
candidate_df.head(10)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,cart_size,product_id,target,user_total_orders,user_avg_basket_size,user_avg_days_between_orders,user_avg_order_hour,user_product_orders,user_product_reorders,user_product_first_order,user_product_last_order,user_product_reorder_rate
0,1,112108,4,4,10,9.0,4,40706,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
1,1,112108,4,4,10,9.0,4,24964,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
2,1,112108,4,4,10,9.0,4,27845,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
3,1,112108,4,4,10,9.0,4,44359,0,3,7.0,11.0,15.0,2.0,1.0,1.0,2.0,0.5
4,1,112108,4,4,10,9.0,4,47626,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
5,1,112108,4,4,10,9.0,4,5707,0,3,7.0,11.0,15.0,2.0,1.0,2.0,3.0,0.5
6,1,112108,4,4,10,9.0,4,21903,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
7,1,112108,4,4,10,9.0,4,45007,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0
8,1,112108,4,4,10,9.0,4,42001,0,3,7.0,11.0,15.0,1.0,0.0,1.0,1.0,0.0
9,1,112108,4,4,10,9.0,4,21137,0,3,7.0,11.0,15.0,0.0,0.0,0.0,0.0,0.0


In [154]:
dim_products = pd.read_csv("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/raw/dim_products.csv")

In [155]:
# adding product level info 

candidate_df = candidate_df.merge(dim_products, on="product_id", how="left")

In [156]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target',
       'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour',
       'user_product_orders', 'user_product_reorders',
       'user_product_first_order', 'user_product_last_order',
       'user_product_reorder_rate', 'product_name', 'aisle_id',
       'department_id'],
      dtype='str')

In [157]:
candidate_df["has_bought_before"] = (candidate_df["user_product_orders"] > 0).astype(int)
candidate_df["has_reordered"] = (candidate_df["user_product_reorders"] > 0).astype(int)

In [158]:
candidate_df.columns

Index(['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
       'days_since_prior_order', 'cart_size', 'product_id', 'target',
       'user_total_orders', 'user_avg_basket_size',
       'user_avg_days_between_orders', 'user_avg_order_hour',
       'user_product_orders', 'user_product_reorders',
       'user_product_first_order', 'user_product_last_order',
       'user_product_reorder_rate', 'product_name', 'aisle_id',
       'department_id', 'has_bought_before', 'has_reordered'],
      dtype='str')

In [159]:
candidate_df.to_parquet("/Users/abhijeetsinghparihar/Desktop/Projects/Supply Chain Project/CartIQ/data/processed/model_data.parquet", index=False)